## <a name="train_mask_rcnn_with_simple_shape_dataset">7. 训练Mask R-CNN模型</a>


在这一部分实践中，我们结合预训练模型，使用最小COCO数据集来训练Mask R-CNN模型。

原始的COCO数据集非常庞大，从头训练需要耗费较长的时间，因此我们加载预训练模型，然后用一个约500张图片的mini数据集对Mask R-CNN模型进行训练。mini数据集对模型参数的影响非常小，这部分内容主要让大家对模型训练的整个过程有一个了解。在学习完模型训练过程后，大家可以尝试在算力较强的环境中从头训练模型，也可以尝试使用其他数据集对已有的与训练模型进行迁移学习的实践。

本节所使用的代码都已经封装到src目录下，具体的代码逻辑可以参考src目录下的源文件。


第一步：导入相应的Python库，准备预训练模型

In [ ]:
import sys
import random
import math
import re
import time
import numpy as np
import cv2
import matplotlib
import matplotlib.pyplot as plt

from src.mrcnn.config import Config
from src.mrcnn import utils
import src.mrcnn.model as modellib
from src.mrcnn import visualize
from src.mrcnn.model import log

%matplotlib inline 

# Directory to save logs and trained model
MODEL_DIR = "logs"

# Local path to trained weights file
COCO_MODEL_PATH = "data/mask_rcnn_coco.h5"

第二步：生成相关配置项

我们定义Config类的子类MyTrainConfig，指定相关的参数，较为关键的参数有：


- __NAME__： Config的唯一名称
- __NUM_CLASSIS__： 分类的数量，我们只生成圆形，正方形和三角形，再加上背景，因此一共是4个分类
- __IMAGE_MIN_DIM和IMAGE_MAX_DIM__： 图片的最大和最小尺寸，我们生成固定的128x128的图片，因此都设置为128
- __TRAIN_ROIS_PER_IMAGE__： 每张图片上训练的RoI个数
- __STEPS_PER_EPOCH和VALIDATION_STEPS__： 训练和验证时，每轮的step数量，减少step的数量可以加速训练，但是检测精度降低


In [ ]:
class MyTrainConfig(Config):
    # 可辨识的名称
    NAME = "my_train"

    # GPU的数量和每个GPU处理的图片数量，可以根据实际情况进行调整，参考为Nvidia Tesla P100
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1

    # 物体的分类个数，COCO中共有80种物体+背景
    NUM_CLASSES = 1 + 80  # background + 80 shapes

    # 图片尺寸统一处理为1024，可以根据实际情况再进一步调小
    IMAGE_MIN_DIM = 1024
    IMAGE_MAX_DIM = 1024

    # 因为我们生成的形状图片较小，这里可以使用较小的Anchor进行RoI检测
#     RPN_ANCHOR_SCALES = (8, 16, 32, 64, 128)  # anchor side in pixels

    # 每张图片上训练的RoI个数，因为我们生成的图片较小，而且每张图片上的形状较少
    #　因此可以适当调小该参数，用较少的Anchor即可覆盖大致的物体信息
    TRAIN_ROIS_PER_IMAGE = 200

    # 每轮训练的step数量
    STEPS_PER_EPOCH = 100

    # 每轮验证的step数量
    VALIDATION_STEPS = 20
    
config = MyTrainConfig()
config.display()

第三步：准备数据集

我们使用封装好的CocoDataset类，生成训练集和验证集。

In [ ]:
from src.mrcnn.coco import CocoDataset
COCO_DIR = 'data'

# 生成训练集
dataset_train = CocoDataset()
dataset_train.load_coco(COCO_DIR, "train") #　加载训练数据集
dataset_train.prepare()


# 生成验证集
dataset_val = CocoDataset()
dataset_val.load_coco(COCO_DIR, "val") #　加载验证数据集
dataset_val.prepare()

## 创建模型

第一步：用"training"模式创建模型对象，用于形状数据集的训练

In [ ]:
model = modellib.MaskRCNN(mode="training", config=config,
                          model_dir=MODEL_DIR)

第二步：加载预训练模型的权重

In [ ]:
model.load_weights(COCO_MODEL_PATH, by_name=True)

接下来，我们使用预训练的模型，结合Shapes数据集，对模型进行训练

## 训练模型

Keras中的模型可以按照制定的层进行构建，在模型的train方法中，我们可以通过layers参数来指定特定的层进行训练。layers参数有以下几种预设值：

- heads:只训练head网络中的分类、mask和bbox回归
- all: 所有的layer
- 3+: 训练ResNet Stage3和后续Stage
- 4+: 训练ResNet Stage4和后续Stage
- 5+: 训练ResNet Stage5和后续Stage

此外，layers参数还支持正则表达式，按照匹配规则指定layer，可以调用model.keras_model.summary()查看各个层的名称，然后按照需要指定要训练的层。

In [ ]:
model.train(dataset_train, dataset_val, 
            learning_rate=config.LEARNING_RATE, 
            epochs=1, 
            layers='all')

model_savepath = 'my_mrcnn_model.h5'
model.keras_model.save_weights(model_savepath)

## 使用模型检测图片物体

第一步：定义InferenceConfig，并创建"Inference"模式的模型对象

In [ ]:
class InferenceConfig(MyTrainConfig):
    GPU_COUNT = 1
    IMAGES_PER_GPU = 1

inference_config = InferenceConfig()

inference_model = modellib.MaskRCNN(mode="inference", 
                                    config=inference_config,
                                    model_dir=MODEL_DIR)

# 因为我们在上一步训练出了模型，kears自动保存模型文件
# 我们可以调用find_last()方法找到最后一次训练出的模型文件
# model_path = model.find_last()
# 也可以手动指定训练好的模型文件地址
# model_path = os.path.join(ROOT_DIR, ".h5 file name here")

# 加载我们自己训练出的形状模型文件的权重信息
# print("Loading weights from ", model_path)
inference_model.load_weights(COCO_MODEL_PATH, by_name=True)
# inference_model.load_weights(model_savepath, by_name=True)

第二步：从验证数据集中随机选出一张图片进行预测，并显示结果

In [ ]:
# 随机选出图片进行测试
image_id = random.choice(dataset_val.image_ids)
original_image, image_meta, gt_class_id, gt_bbox, gt_mask =\
    modellib.load_image_gt(dataset_val, inference_config, 
                           image_id, use_mini_mask=False)

log("original_image", original_image)
log("image_meta", image_meta)
log("gt_class_id", gt_class_id)
log("gt_bbox", gt_bbox)
log("gt_mask", gt_mask)

det_instances_savepath = '07.det_instances.jpg'
visualize.display_instances(original_image, gt_bbox, gt_mask, gt_class_id, 
                            dataset_train.class_names, figsize=(8, 8),
                           save_path=det_instances_savepath)

In [ ]:
# 定义助手函数用于设置matplot中的子绘制区域所在的行和列
def get_ax(rows=1, cols=1, size=8):
    
    _, ax = plt.subplots(rows, cols, figsize=(size*cols, size*rows))
    return ax

results = inference_model.detect([original_image], verbose=1)

r = results[0]
prediction_savepath = '07.prediction.jpg'
visualize.display_instances(original_image, r['rois'], r['masks'], r['class_ids'], 
                            dataset_val.class_names, r['scores'], ax=get_ax(),
                            save_path=prediction_savepath)

## 评估模型

这一步我们对自己训练出的模型进行一个简单的评估。计算模型的平均精度mAP（mean Average Precision)

In [ ]:
# 计算VOC类型的 mAP，条件是 IoU=0.5
# 下面的示例中只选出10张图片进行评估，增加图片数量可以增加模型评估的准确性
image_ids = np.random.choice(dataset_val.image_ids, 10)
APs = []
for image_id in image_ids:
    # Load image and ground truth data
    image, image_meta, gt_class_id, gt_bbox, gt_mask =\
        modellib.load_image_gt(dataset_val, inference_config,
                               image_id, use_mini_mask=False)
    molded_images = np.expand_dims(modellib.mold_image(image, inference_config), 0)
    # Run object detection
    results = inference_model.detect([image], verbose=0)
    r = results[0]
    # Compute AP
    AP, precisions, recalls, overlaps =\
        utils.compute_ap(gt_bbox, gt_class_id, gt_mask,
                         r["rois"], r["class_ids"], r["scores"], r['masks'])
    APs.append(AP)
    
print("mAP: ", np.mean(APs))

至此，我们就完成了自有模型的训练，并且理解了整个Mask R-CNN模型的训练过程。如果条件允许，大家可以尝试去掉预训练模型的加载，从头开始训练模型，并调整MyTrainConfig中的参数来优化模型的效果。此外，我们也可以从网络中收集新的图片对模型进行验证、调优。在理解Mask R-CNN的基础上，我们还可以用Mask R-CNN进行更高层的应用，例如[《Mask R-CNN》论文](https://arxiv.org/abs/1703.06870)中提到的人体姿势识别。Mask R-CNN是一个非常优秀的模型，希望通过对Mask R-CNN的学习，我们对多任务模型有了一定的理解，在计算机视觉、机器学习领域还有更多新的研究成果，等待大家去探索。